In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)
# scale to [0, 1]
X_trainval, X_test, y_trainval, y_test = train_test_split(
X, y, test_size=10000, random_state=42, stratify=y
)


In [2]:
import os
from torch import nn

class MLP_Model(nn.Module):
    def __init__(self):
        super(MLP_Model, self).__init__()
        self.input = nn.Linear(784,2 * 784 )
        self.hidden = nn.Linear(2*784, 2*784)
        self.output = nn.Linear(2*784, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input(x))
        x = self.relu(self.hidden(x))
        x = self.output(x)

        return x


os.makedirs('Models', exist_ok=True)

Optuna Objective

In [5]:
import optuna
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import KFold

# --- Dane ---
X_trainval_tensor = torch.from_numpy(X_trainval).float()
y_trainval_tensor = torch.from_numpy(y_trainval).long()
dataset = TensorDataset(X_trainval_tensor, y_trainval_tensor)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS = 5
N_EPOCHS = 5
BATCH    = 1024


def _train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(X), y).backward()
        optimizer.step()

def _calculate_val_loss(model, loader, criterion):
    model.eval()
    with torch.no_grad():
        for X,y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            loss = criterion(output, y)

    return loss.item()


def objective(trial: optuna.Trial) -> float:
    # Parameters

    lr       = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    momentum = trial.suggest_float("momentum", 0.0, 0.99)
    nesterov = trial.suggest_categorical("nesterov", [True, False])
    if momentum == 0.0:
        nesterov = False


    criterion = nn.CrossEntropyLoss()
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_loss = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval)):
        train_loader = DataLoader(
            Subset(dataset, train_idx), batch_size=BATCH, shuffle=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx), batch_size=512
        )

        model = MLP_Model().to(DEVICE)

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            nesterov=nesterov,
        )

        for epoch in range(N_EPOCHS):
            _train_one_epoch(model, train_loader, optimizer, criterion)

        fold_loss.append(_calculate_val_loss(model, val_loader, criterion))

    return float(np.mean(fold_loss))




study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5, timeout=1800, show_progress_bar=True)

print("Best parameters:", study.best_params)
print("Mean avg val loss CV: ", study.best_value)

[I 2026-03-29 19:51:38,341] A new study created in memory with name: no-name-d3c2f117-b9bb-484d-97bd-74d7e964e947


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-03-29 19:51:54,191] Trial 0 finished with value: 2.3001710414886474 and parameters: {'learning_rate': 0.00014667014319556023, 'momentum': 0.15753037705975856, 'nesterov': True}. Best is trial 0 with value: 2.3001710414886474.
[I 2026-03-29 19:52:10,632] Trial 1 finished with value: 0.5568734288215638 and parameters: {'learning_rate': 0.01149356369185051, 'momentum': 0.7486587012769177, 'nesterov': False}. Best is trial 0 with value: 2.3001710414886474.
[I 2026-03-29 19:52:26,771] Trial 2 finished with value: 1.8591806173324585 and parameters: {'learning_rate': 0.005528792969666366, 'momentum': 0.5451778124809656, 'nesterov': False}. Best is trial 0 with value: 2.3001710414886474.
[I 2026-03-29 19:52:42,079] Trial 3 finished with value: 2.286655378341675 and parameters: {'learning_rate': 0.00037250373378859905, 'momentum': 0.582071750451733, 'nesterov': False}. Best is trial 0 with value: 2.3001710414886474.
[I 2026-03-29 19:52:57,166] Trial 4 finished with value: 2.281867980957